In [3]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_excel(r"E:\AAA_projects\Real_estate\pr_5_data_HousePricePrediction.xlsx")

# ==========================================
# 1. Check datatypes of all columns
# ==========================================
print("Datatype Check:")
print(df.dtypes)


Datatype Check:
Id                int64
MSSubClass        int64
MSZoning         object
LotArea         float64
LotConfig        object
BldgType         object
OverallCond     float64
YearBuilt       float64
YearRemodAdd    float64
Exterior1st      object
BsmtFinSF2      float64
TotalBsmtSF     float64
SalePrice       float64
dtype: object


In [4]:
df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')

In [5]:
df.columns

Index(['id', 'mssubclass', 'mszoning', 'lotarea', 'lotconfig', 'bldgtype',
       'overallcond', 'yearbuilt', 'yearremodadd', 'exterior1st', 'bsmtfinsf2',
       'totalbsmtsf', 'saleprice'],
      dtype='object')

In [12]:
df.rename(columns={
    'mssubclass': 'ms_sub_class',
    'mszoning': 'ms_zoning',
    'lotarea': 'lot_area',
    'lotconfig': 'lot_config',
    'bldgtype': 'bldg_type',
    'overallcond': 'overall_cond',
    'yearbuilt': 'year_built',
    'yearremodadd': 'year_remod_add',
    'exterior1st': 'exterior_1st',
    'bsmtfinsf2': 'bsmt_fin_sf_2',
    'totalbsmtsf': 'total_bsmt_sf',
    'saleprice': 'sale_price'
}, inplace=True)

In [13]:
df.columns

Index(['id', 'ms_sub_class', 'ms_zoning', 'lot_area', 'lot_config',
       'bldg_type', 'overall_cond', 'year_built', 'year_remod_add',
       'exterior_1st', 'bsmt_fin_sf_2', 'total_bsmt_sf', 'sale_price'],
      dtype='object')

In [14]:
# Column-wise Description of Dataset
# Column Name	Description
# id	Unique identification number assigned to each property record. It helps distinguish one property from another and is mainly used for reference rather than analysis.
# ms_sub_class	Identifies the type or class of dwelling based on construction style. It represents different residential building categories such as single-family homes, duplexes, or multi-level structures.
# ms_zoning	Indicates the zoning classification of the property area. It shows the type of residential zone where the property is located, such as residential low density or medium density, which influences land usage and market value.
# lot_area	Total land area of the property measured in square feet. Larger lot areas generally increase property value because they provide more space and future utility.
# lot_config	Describes the arrangement or shape of the lot, such as inside lot, corner lot, frontage on two sides, or irregular shape. Lot arrangement affects accessibility and buyer preference.
# bldg_type	Specifies the building type of the property, such as single-family detached house, townhouse, duplex, or other residential structures.
# overall_cond	Represents the overall physical condition of the property on a rating scale. Higher values indicate better structural maintenance and stronger property condition.
# year_built	The year in which the property was originally constructed. It helps measure property age and understand how construction period affects market value.
# year_remod_add	The year when remodeling, renovation, or major additions were made to the property. This helps evaluate modernization impact on sale price.
# exterior_1st	Indicates the main exterior material used for the house, such as vinyl siding, metal siding, brick, or wood. Exterior material influences durability and visual appeal.
# bsmt_fin_sf_2	Basement finished area of secondary section measured in square feet. It shows additional usable finished basement space if available.
# total_bsmt_sf	Total basement area measured in square feet including finished and unfinished sections. Larger basement size often increases usable house value.
# sale_price	Final selling price of the property. This is the target variable used to understand which features influence market price.

In [15]:
# ==========================================
# FEATURE ENGINEERING - NEW COLUMNS
# ==========================================
# 1. Property age
df['property_age'] = 2025 - df['year_built']

# 2. Renovation gap (years between construction and remodel)
df['renovation_gap'] = df['year_remod_add'] - df['year_built']

# 3. Renovation status
df['is_renovated'] = np.where(df['renovation_gap'] > 0, 'Yes', 'No')

# 4. Price per square foot of lot area
df['price_per_sqft'] = df['sale_price'] / df['lot_area']

# 5. Basement utilization ratio
df['basement_ratio'] = df['total_bsmt_sf'] / df['lot_area']

# 6. Property age category
df['age_group'] = pd.cut(
    df['property_age'],
    bins=[0, 10, 30, 60, 200],
    labels=['New', 'Moderate', 'Old', 'Very Old']
)

# 7. Lot size category
df['lot_size_category'] = pd.cut(
    df['lot_area'],
    bins=[0, 5000, 10000, 15000, 100000],
    labels=['Small', 'Medium', 'Large', 'Very Large']
)

In [16]:
df.columns

Index(['id', 'ms_sub_class', 'ms_zoning', 'lot_area', 'lot_config',
       'bldg_type', 'overall_cond', 'year_built', 'year_remod_add',
       'exterior_1st', 'bsmt_fin_sf_2', 'total_bsmt_sf', 'sale_price',
       'property_age', 'renovation_gap', 'is_renovated', 'price_per_sqft',
       'basement_ratio', 'age_group', 'lot_size_category'],
      dtype='object')

In [17]:
# | New Column        | Insight Value               |
# | ----------------- | --------------------------- |
# | property_age      | shows age impact on price   |
# | renovation_gap    | renovation value effect     |
# | is_renovated      | direct market comparison    |
# | price_per_sqft    | value efficiency            |
# | basement_ratio    | land use efficiency         |
# | age_group         | buyer segment understanding |
# | lot_size_category | market segmentation         |


In [18]:
#Price Behavior by Age Group
print(df.groupby('age_group')['sale_price'].agg(['mean', 'median', 'count']))
#Shows whether newer homes always cost more.

                    mean    median  count
age_group                                
New                  NaN       NaN      0
Moderate   240796.485417  219500.0    480
Old        170846.606880  156000.0    407
Very Old   137208.065603  132500.0    564


C:\Users\ahama\AppData\Local\Temp\ipykernel_18040\4216457152.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby('age_group')['sale_price'].agg(['mean', 'median', 'count']))


In [19]:
#Renovated vs Non-Renovated Market Difference
print(df.groupby('is_renovated')['sale_price'].agg(['mean', 'median']))
# Why 
# Direct renovation premium.

                       mean    median
is_renovated                         
No            182808.036794  170000.0
Yes           178819.297101  154950.0


In [20]:
#Lot Size Category Efficiency
print(df.groupby('lot_size_category')['price_per_sqft'].mean())
#Large plots may have lower price efficiency.

lot_size_category
Small         47.261891
Medium        19.492015
Large         17.798550
Very Large    12.111064
Name: price_per_sqft, dtype: float64


C:\Users\ahama\AppData\Local\Temp\ipykernel_18040\2221208806.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby('lot_size_category')['price_per_sqft'].mean())


In [24]:
df.to_excel(r'E:\AAA_projects\Real_estate\real_estate_cleaned.xlsx', index=False)

In [26]:
!pip install mysql-connector-python pymysql sqlalchemy

import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus

# Load cleaned file
df = pd.read_excel(r"E:\AAA_projects\Real_estate\real_estate_cleaned.xlsx")

# MySQL connection
username = "root"
password = quote_plus("Password@123")
host = "127.0.0.1"
port = "3306"
database = "real_es"

engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}"
)

# Upload to MySQL
df.to_sql("real_estate", engine, if_exists="replace", index=False)

# Verify upload
pd.read_sql("SELECT * FROM real_estate LIMIT 5;", engine)

,id,ms_sub_class,ms_zoning,lot_area,lot_config,bldg_type,overall_cond,year_built,year_remod_add,exterior_1st,bsmt_fin_sf_2,total_bsmt_sf,sale_price,property_age,renovation_gap,is_renovated,price_per_sqft,basement_ratio,age_group,lot_size_category
0,0,60,RL,8450.0,Inside,1Fam,5.0,2003.0,2003.0,VinylSd,0.0,856.0,208500.0,22.0,0.0,No,24.674556,0.101302,Moderate,Medium
1,1,20,RL,9600.0,FR2,1Fam,8.0,1976.0,1976.0,MetalSd,0.0,1262.0,181500.0,49.0,0.0,No,18.906250,0.131458,Old,Medium
2,2,60,RL,11250.0,Inside,1Fam,5.0,2001.0,2002.0,VinylSd,0.0,920.0,223500.0,24.0,1.0,Yes,19.866667,0.081778,Moderate,Large
3,3,70,RL,9550.0,Corner,1Fam,5.0,1915.0,1970.0,Wd Sdng,0.0,756.0,140000.0,110.0,55.0,Yes,14.659686,0.079162,Very Old,Medium
4,4,60,RL,14260.0,FR2,1Fam,5.0,2000.0,2000.0,VinylSd,0.0,1145.0,250000.0,25.0,0.0,No,17.531557,0.080295,Moderate,Large
